In [ ]:
import pandas as pd

# Path to data frame containing all WSIs with a matched rekvnr
df_path = "D:\DATA\overlapping_rekvnr.xlsx"
df_overlapping = pd.read_excel(df_path)

In [ ]:
print(df_overlapping.head())

In [ ]:
# Overview of all missing values
missing_counts = df_overlapping.isnull().sum()
print("\nMissing values: \n", missing_counts)

In [ ]:
# Path to SNOMED codes (all codes with code history)
snomed_path = "D:/DATA/patoSnoMed_2025-04.xlsx"

xls_snomed = pd.read_excel(snomed_path)
print(f"Columns: {xls_snomed.columns.tolist()}")

In [ ]:
# Data Frame with relevant columns
df_snomed = pd.DataFrame(xls_snomed, columns=['SKSkode', 'DatoFra', 'DatoÆndring', 'DatoTil', 'Kodetekst', 'Fuldtekst'])

print(df_snomed.head())

In [ ]:
# Convert SNOMED codes to a set for fast lookup
snomed_set = set(df_snomed['SKSkode'].dropna().astype(str))

# Function to get first letters of SNOMED codes
def get_first_letters(codes):
    """ Given a list of SNOMED codes, return a list of unique first letters. """
    return {code[0] for code in codes if code}

# Get unique first letters
first_letters = get_first_letters(df_snomed['SKSkode'].dropna().unique())
print('SNOMED letters:', first_letters)


In [ ]:
# Code to Find Valid SNOMED Codes in "snomed kode" column
# All valid codes (with history) in snomed_set

import re

# Function to extract unique valid snomed codes by first letter
def extract_by_letter(cell, letter, valid_codes):
    if pd.isnull(cell):
        return []
    # Split string by common separators
    parts = re.split(r"[ ,;]+", cell)
    # Keep only unique, valid codes that start with the desired letter
    codes = {part for part in parts if part in valid_codes and part.startswith(letter)}
    return list(codes) 

# Define which letters to treat separately
main_letters = ["T", "M"]
other_letters = list(first_letters - set(main_letters))

# Apply for T and M
for letter in main_letters:
    df_overlapping[letter] = df_overlapping["snomed kode"].apply(lambda x: extract_by_letter(x, letter, snomed_set))

# Apply for all other letters and combine into one "Other" column
def extract_other(cell, main_letters, valid_codes):
    if pd.isnull(cell):
        return []
    parts = re.split(r"[ ,;]+", cell)
    codes = {part for part in parts if part in valid_codes and part[0] not in main_letters}
    return list(codes)

df_overlapping["Other"] = df_overlapping["snomed kode"].apply(lambda x: extract_other(x, main_letters, snomed_set))

# Check result
print(df_overlapping.head())

In [ ]:
# Function to check for missing values or empty lists
def is_missing(x):
    try:
        if isinstance(x, list):
            return len(x) == 0
        elif isinstance (x, tuple):
            return len(x) == 0
        return pd.isnull(x)
    except Exception:
        return False

# Check column T and M
missing_mask_T = df_overlapping["T"].apply(is_missing)
missing_mask_M = df_overlapping["M"].apply(is_missing)

print("Number of missing values in T:", missing_mask_T.sum())
print("Number of missing values in M:", missing_mask_M.sum())

In [ ]:
# Code to find missing T and M codes in text columns, using valid codes set

text_columns = ["makrotekst", "mikrotekst", "kode fritekst"]

def find_missing_codes(row, letter, columns, valid_codes):
    if is_missing(row[letter]):
        found = []
        for c in columns: 
            codes = extract_by_letter(row[c], letter, valid_codes)
            if codes: 
                found.extend(codes)
        return list(set(found)) if found else None
    return None

# Apply for T and M
for letter in main_letters:
    df_overlapping[f"{letter} candidates"] = df_overlapping.apply(lambda r: find_missing_codes(r, letter, text_columns, snomed_set), axis=1)

# Count rows where T and M is missing but candidates found
fixable_T_count = df_overlapping["T candidates"].notna().sum()
fixable_M_count = df_overlapping["M candidates"].notna().sum()

print("Number of fixable T rows:", fixable_T_count)
print("Number of fixable M rows:", fixable_M_count)

In [ ]:
fixable_T = df_overlapping[df_overlapping["T candidates"].notna()]
print("\nFixable T rows: \n", fixable_T['T candidates'])

fixable_M = df_overlapping[df_overlapping["M candidates"].notna()]
print("\nFixable M rows: \n", fixable_M['M candidates'])

In [ ]:
# Code to fill SNOMED codes from text columns

def fill_from_candidates(row, col, valid_codes):
    current = row[col]
    candidates = row[f"{col} candidates"]

    # Check if current is missing (None or empty list)
    if is_missing(current):
        if candidates is not None:
            # Ensure unique + valid
            return list({c for c in candidates if c in valid_codes})
        return []
    return current

for letter in main_letters: 
    df_overlapping[letter] = df_overlapping.apply(lambda r: fill_from_candidates(r, letter, snomed_set), axis=1)


In [ ]:
# Check missing values again
missing_mask_T = df_overlapping["T"].apply(is_missing)
missing_mask_M = df_overlapping["M"].apply(is_missing)

print("Number of missing values in T:", missing_mask_T.sum())
print("Number of missing values in M:", missing_mask_M.sum())


In [ ]:
# Duplicate Rows

# Convert lists to tuples
df_tuples = df_overlapping.map(
    lambda x: tuple(x) if isinstance(x, list) else x  
)

print("Number of duplicated rows:", df_tuples.duplicated().sum())

In [ ]:
# Duplicate values in rekvnr

# Mask for rows with duplicate rekvnr
dup_mask = df_tuples["rekvnr"].duplicated(keep=False)
df_dups = df_tuples[dup_mask]

num_dups = df_tuples["rekvnr"].duplicated().sum()
print("Number of duplicate rekvnr:", num_dups)

In [ ]:
# Show rows with duplicate rekvnr
df_dups_sorted = df_dups.sort_values("rekvnr")

print(df_dups_sorted)

In [ ]:
def combine_values(series):
    """ Combine differing values in a column into a list of unique values. """
    # If all values are the same, return the single value
    uniques = series.dropna().unique()
    if len(uniques) == 1:
        return uniques[0]
    else:
        return list(uniques)

# Combine rows with same rekvnr and modtdato
df_no_dups = df_tuples.groupby(["rekvnr", "modtdato"], as_index=False).agg(combine_values)


In [ ]:
# Get results
print("Before combing duplicated rekvnr:", len(df_tuples))
print("After combining rows with duplicate rekvnr:", len(df_no_dups))

In [ ]:
# Check missing values in all columns
for col in df_no_dups.columns: 
    missing_count = df_no_dups[col].apply(is_missing).sum()
    print(f"{col}: ", missing_count)

In [ ]:
# Get number of files with missing T or M codes

# Step 1: Find rows with missing T or M
rows_with_missing = df_no_dups[df_no_dups['T'].apply(is_missing) | df_no_dups['M'].apply(is_missing)]

# Step 2: Count total number of filenames in those rows
unique_files_missing = set().union(*rows_with_missing['wsi filenames'])
print("Total number of unique files with missing T or M:", len(unique_files_missing))

# Step 3: Remove those rows from df_wsi
df_clean = df_no_dups.drop(rows_with_missing.index)


In [ ]:
# Get results
print("Before removing missing values:", len(df_no_dups))
print("After removing rows with missing T or M:", len(df_clean))

In [ ]:
# Remove candidate columns
cols_to_drop = ["T candidates", "M candidates"]
df_clean = df_clean.drop(columns=cols_to_drop)

In [ ]:
# Check correct values / formatting
print(df_clean.head(3))

In [ ]:
# Check correct formatting
def get_pattern(col):
    if col == "rekvnr": 
        return r"\d{8}"           # 8 digits
    elif col == "modtdato" or col == "rekvdato":
        return r"\d{4}-\d{2}-\d{2}"  # YYYY-MM-DD
    elif col == "team":
        return r"[A-ZÆØÅ]{3}"           # 3 uppercase letters
    elif col == "sex":
        return r"[FMfm]"               # F or M
    elif col == "alder" or col == "matantal" or col == "mattype" or col == "wsi count":
        return r"\d+"             # digits
    else:
        return None

def is_correct(value, pattern):
    if pattern is None or pd.isna(value):
        return False
    return bool(re.fullmatch(pattern, str(value).strip()))


In [ ]:
for col in df_clean.columns: 
    pattern = get_pattern(col)
    if pattern is None:
        continue
    incorrect = (~df_clean[col].apply(lambda x: is_correct(x, pattern))).sum()
    print(f"Number of {col} with incorrect formatting:", incorrect)

In [ ]:
# Show all unique values in the 'sex' column
unique_sex_values = df_clean['sex'].unique()
print(unique_sex_values)

# Optionally, count occurrences of each value
sex_counts = df_clean['sex'].value_counts(dropna=False)
print(sex_counts)


In [ ]:
# Standardize sex column to uppercase F or M
df_clean['sex'] = df_clean['sex'].apply(lambda x: str(x).upper() if pd.notna(x) else x)

# Check unique values
print(df_clean['sex'].unique())


In [ ]:
# Show all unique values in the 'team' column
unique_team_values = df_clean['team'].unique()
print(unique_team_values)

# Optionally, count occurrences of each value
team_counts = df_clean['team'].value_counts(dropna=False)
print(team_counts)


In [ ]:
for col in df_clean.columns: 
    print(f"{col}: :", type(col))

In [ ]:
# CHECK DATES

# Convert to datetime if not already
df_clean['modtdato'] = pd.to_datetime(df_clean['modtdato'], errors='coerce')
df_clean['rekvdato'] = pd.to_datetime(df_clean['rekvdato'], errors='coerce')

# Only allows dates from 2011 - 2013
mask_valid_dates = (df_clean['modtdato'].dt.year >= 2011) & (df_clean['modtdato'].dt.year <= 2013)
invalid_dates = df_clean[~mask_valid_dates]
print("Rows with modtdato outside 2011-2013:", len(invalid_dates))

# MODTDATO must be after REKVDATO
mask_modtdato_before_rekvdato = df_clean['modtdato'] < df_clean['rekvdato']
invalid_modtdato = df_clean[mask_modtdato_before_rekvdato]
print("Rows with modtdato later than rekvdato:", len(invalid_modtdato))


In [ ]:
# CHECK NUMERIC COLUMNS

# Age check
invalid_age = df_clean[(df_clean['alder'] < 0) | (df_clean['alder'] > 120)]
print("Rows with invalid age:", len(invalid_age))

# matantal and wsi count should be >= 0
invalid_matantal = df_clean[df_clean['matantal'] < 0]
invalid_wsi_count = df_clean[df_clean['wsi count'] < 0]
print("Rows with negative matantal:", len(invalid_matantal))
print("Rows with negative wsi count:", len(invalid_wsi_count))

In [ ]:
print(df_clean.head())
print("\nData frame shape: ", df_clean.shape)


In [ ]:
# Save to Excel
output_file = "D:\DATA\df_cleaned.xlsx"
df_clean.to_excel(output_file, index=False)

print(f"Saved DataFrame to {output_file}")